# QUERY 4
Cuentas que cumplan con el patrón scatter-gather con una sola cuenta de separación,
para cuentas que hayan realizado transferencias en USD hacia 5 cuentas distintas dentro
del período [2022-09-01, 2022-09-05] (alineado exactamente con el Notebook Cátedra)

In [ ]:
import numpy as np
import pandas as pd

# 1. Cargar datos
RUTA = "../../datasets"
DATASET = "HI-Medium_Trans_sg.csv"
ACCOUNT = "HI-Medium_accounts.csv"
NOMBRE_ARCHIVO_SOLUCION = "q4_solucion.csv"

trans_df = pd.read_csv(f"{RUTA}/{DATASET}")
accounts_df = pd.read_csv(f"{RUTA}/{ACCOUNT}")
print("SIZE trans_df:", trans_df.size)
print("SIZE accounts_df:", accounts_df.size)

SIZE trans_df: 76164671
SIZE accounts_df: 3563440


In [5]:
# 2. Filtrar transacciones en USD y período
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE trans_usd_sept_1st_df:", trans_usd_sept_1st_df.shape[0])

SIZE trans_usd_sept_1st_df: 1393082


In [6]:
# 3. Filtrar cuentas scatter-calificadas (A envió a > 5 destinos distintos)
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size > 5)
print("SIZE ranged_trans_usd_sept_df:", ranged_trans_usd_sept_df.shape[0])

# 4. Aristas A→B y B→C
accounts_ab = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
accounts_bc = trans_usd_sept_1st_df[["From Bank", "Account", "To Bank", "Account.1"]]

# Merge para encontrar caminos A→B→C
account_pairs_df = accounts_ab.merge(accounts_bc, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})

# Evitar ciclos de longitud 2 (A != C)
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]

# Deduplicar triplets (A, B, C) para no contar el mismo intermediario más de una vez
account_pairs_df = account_pairs_df.drop_duplicates(subset=["From Bank", "From Account", "To Bank_x", "Account.1_x", "To Bank", "To Account"])

# Agrupar y filtrar donde más de 5 intermediarios conectan A y C
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]

# Formatear resultado final
from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts.to_csv(NOMBRE_ARCHIVO_SOLUCION, index=False)
print(unique_accounts)

SIZE ranged_trans_usd_sept_df: 406050
          Bank          Account
132408    1394        800701D70
226641  990001  SG_SRC_04D9A303
132408   11495        800A08E20
226641  990099  SG_DST_04D9A303
